## Getting started using the UK Biobank pVCF data

**Overview:** <br>

This notebook shows several ways of working with population VCFs (pVCFs), including handling variants with very long identifiers and multiallelic sites, using the [Swiss Army Knife](https://ukbiobank.dnanexus.com/app/swiss-army-knife) tool. 

When working with UK Biobank (UKB) pVCF data, there are some considerations. Some variants in the pVCF files may have very long variant IDs. At certain genomic positions, these IDs can exceed plink/plink2’s 16,000-character limit for variant identifiers, resulting in errors. In addition, some pVCFs contain multiallelic records, where variants sharing the same POS field are merged into a single entry rather than represented as separate biallelic records.

With these considerations in mind this notebook demonstrates how to:

1) Carry out basic manipulations on pVCF files using bcftools. An example of a manipulation that can be performed using bcftools is filtering for sample IDs, genomic regions and/or genomic variants. See http://samtools.github.io/bcftools/bcftools.html for more information on the capabilities of bcftools.

2) Normalise variants and split multiallelic sites into multiple rows using bcftools.

3) Reformat the variant ID column to contain shorter IDs, or no IDs, enabling downstream use of plink/plink2.  

4) Convert the pVCF(s) to plink2 binary format (.pgen/.pvar/.psam).

5) Convert the pVCF(s) to BGEN format (.bgen/.sample).

6) Submit jobs for multiple input pVCF files. 

7) Merge plink2 files for each pVCF.

Throughout this notebook we will use the BEAGLE data (Field 30108) as an example data set. The 500k BEAGLE Phased VCFs are available on the UKB-RAP at /Bulk/DRAGEN WGS/BEAGLE Phased VCFs [500k release]/. These data are described on Showcase in [Field 30108](https://biobank.ndph.ox.ac.uk/showcase/field.cgi?id=30108) and in the [500k BEAGLE Phased VCFs data release](https://community.ukbiobank.ac.uk/hc/en-gb/articles/31907475048733-500k-BEAGLE-Phased-VCFs-data-release) article. 

**Overall Compute Information:** <br>
Running jobs non-interactively is often more cost-effective. [Swiss Army Knife](https://ukbiobank.dnanexus.com/app/swiss-army-knife) is a prebuilt applet provided by DNAnexus.  Whilst Swiss Army Knife is used in these instances, it is possible to build your own applets using dx-app-wizard.

Each operation in this notebook is performed by launching a Swiss Army Knife job via code chunks. The overall runtime and cost estimates for the jobs executed are:

- Runtime: ~1d 5h
- Estimated cost: ~£4-5

Please note that these estimates are approximate, as they are based on phased chromosome 11 pVCFs and may vary depending on chromosome size. Per-job approximate runtime, approximate cost, and instance type information are provided throughout this notebook.

#### 1. Using bcftools to perform basic manipulations

bcftools can be used to filter VCF file(s) for specific sample IDs, genomic positions and/or genomic variants. For example, if we are interested in the phased genotypes of all genomic variants located on chromosome 11q13 (base pair 71,164,008-72,309,374) for a specified set of 1,000 samples, then we can run the code below to filter the VCF to this genomic region and 1,000 samples.

In [ ]:
# first we can use dx toolkit to find the file path for our files of interest
# in this case we want to use the 500k Beagle phased data (field 30108) associated with chromosome 11
dx find data --property field_id=30108 --name "ukb30108_c11*" --delimiter ";" 

# let's save the file paths of these files (column 4) for use in downstream jobs
dx find data --property field_id=30108 --name "ukb30108_c11*" --delimiter ";"  | cut -d';' -f4 > file_list.txt

In [ ]:
# generate list of 1,000 sample IDs to filter for
# here we generate random list of sample IDs but these could already be pre-selected/specified in another file 

# create variable for mount path to .vcf.gz file we are going to extract sample IDs from
VCF=$(grep '.vcf.gz$' file_list.txt | sed 's|^|/mnt/project|')

zgrep -m 1 '#CHROM' "${VCF}" | tr '\t' '\n' | tail -n +10 | grep -v '^W' | shuf -n1000 > random_1k_samples 

In [ ]:
# upload random_1k_samples to project so that it can be used by Swiss Army Knife
# here we upload to a folder in the project called helper_files
# note this folder must already exist in your project and can be created using dx mkdir helper_files

dx upload random_1k_samples --path helper_files/

In [ ]:
# define output folder 
OUTPUT_FOLDER="output_folder/"

In [ ]:
# as we are filtering for genomic region make sure that an index (.tbi) for chromosome 11 is present 
# if an index file does not exist then use the code below to launch a Swiss Army Knife job to generate the .tbi file

VCF_PATH=$(grep ".vcf.gz$" file_list.txt)
echo "VCF_PATH: ${VCF_PATH}"

VCF_FILE=$(basename "${VCF_PATH}")
echo "VCF_FILE: ${VCF_FILE}"

dx run swiss-army-knife \
    -iin="${VCF_PATH}" \
    -icmd="tabix -p vcf ${VCF_FILE}" \
    --name="tabix_index_${VCF_FILE}" \
    --instance-type="mem1_hdd1_v2_x4" \
    --destination="${OUTPUT_FOLDER}" \
    --yes --brief --priority high

In the ```dx run``` command shown above, we specify the input file using the ```-iin``` flag and the command to be executed by the Swiss Army Knife app using the ```-icmd flag```. To make it easier to find, monitor, and organise executions of apps or workflows, it is good practice to use the ```--name``` argument to assign each job a clear and descriptive name. In the example above, the job name reflects both the purpose of the task and the input file being processed.

The ```--instance-type``` option was selected based on the size of the input data (151.24 GB) and the cost and performance information provided in the [UKB-RAP rate card](https://20779781.fs1.hubspotusercontent-na1.net/hubfs/20779781/Product%20Team%20Folder/Rate%20Cards/BiobankResearchAnalysisPlatform_Rate%20Card_Current.pdf). Users can choose between spot and on-demand instances depending on whether cost efficiency or job reliability is a higher priority for their analysis. High-priority jobs are recommended for workloads that need to run immediately and without interruption, typically using on-demand instances. For normal-priority jobs, the Platform first requests spot VMs from EC2 and waits up to 15 minutes for these to become available. If spot VMs are allocated within that time frame, the job will run on them, although execution may be interrupted. If no spot VMs become available within 15 minutes, the Platform automatically requests on-demand VMs instead. For low-priority jobs, the Platform always requests spot VMs. These jobs are the most cost-efficient but can be interrupted. This can be controlled using the ```--priority``` argument, which accepts the values low, normal, or high.

The ```--destination``` argument defines the folder where output files will be stored, the ```--yes``` flag automatically confirms prompts without user input, and the ```--brief``` flag displays a concise output, typically returning one DNAnexus ID per line.

For more information, please see the [Index of dx commands](https://documentation.dnanexus.com/user/helpstrings-of-sdk-command-line-utilities) and [Job Priority](https://dnanexus.gitbook.io/uk-biobank-rap/working-on-the-research-analysis-platform/managing-jobs/managing-job-priority) pages.
                    
**Runtime:** ~1d 3h <br>
**Cost:** ~£4

In [ ]:
# filter for sample IDs and genomic positions using bcftools
VCF_PATH=$(grep ".vcf.gz$" file_list.txt)
echo "VCF_PATH: ${VCF_PATH}"

VCF_PATH_TBI=$(grep ".vcf.gz.tbi$" file_list.txt)
echo "VCF_TBI_PATH: ${VCF_PATH_TBI}"

VCF_FILE=$(basename "${VCF_PATH}")
echo "VCF_FILE: ${VCF_FILE}"

OUTPUT_PREFIX=$(basename "${VCF_PATH}" .vcf.gz)
echo "OUTPUT_PREFIX: ${OUTPUT_PREFIX}"

dx run swiss-army-knife \
        -iin="${VCF_PATH}" \
        -iin="${VCF_PATH_TBI}" \
        -iin="helper_files/random_1k_samples" \
        -icmd="bcftools view -S random_1k_samples -r chr11:71164008-72309374 ${VCF_FILE} -Oz -o ${OUTPUT_PREFIX}.filtered_samples_regions.vcf.gz" \
        --name="bcftools_filter_${VCF_FILE}" \
        --instance-type="mem1_ssd1_v2_x8" \
        --destination="${OUTPUT_FOLDER}" \
        --yes --brief --priority high 

**Runtime:** ~1h 15m <br> 
**Cost:** ~£0.3

#### 2. Variant normalisation and splitting multiallelic variants

The representation of variants in VCF files can vary depending on how they were generated. To ensure consistency across datasets it is good practice to normalise variants against a reference genome. This process involves left-aligning indels and trimming redundant bases to represent each variant in its most compact form. Normalisation helps avoid mismatches when comparing or merging datasets, ensures compatibility with downstream tools, and makes variant representations unambiguous.

Another thing to note is that in the process of generating the phased pVCFs, all records in the VCF file that had the same genomic position, i.e. the POS field, were merged into a single entry. This results in multiallelic sites, entries with more than one alternate allele, rather than individual biallelic records. However, a number of tools require biallelic sites rather than multiallelic sites, e.g. plink/plink2 for association testing and PCA. 

To address the points above bcftools can be used to normalise the variants against a reference genome and split multiallelic sites into biallelic records. Here we use the GRCh38 reference genome, which can be found in your UKB-RAP project at "/Bulk/Exome sequences/Exome OQFE CRAM files/helper_files/" or can be downloaded from:
https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/technical/reference/GRCh38_reference_genome/. If downloading make sure to download both the .fa (fasta) file and the corresponding .fai index file.

In [ ]:
# take our filtered chromosome 11 and carry out variant normalisation and split multiallelic sites
# normalise against GRCh38_full_analysis_set_plus_decoy_hla.fa

# output from previous jobs stored in output_folder/
VCF_PATH=$(dx find data --name="*.filtered_samples_regions.vcf.gz" --path="output_folder" --delimiter=";" | cut -f4 -d ';') 
echo "VCF_PATH: ${VCF_PATH}"

VCF_FILE=$(basename ${VCF_PATH})
echo "VCF_FILE: ${VCF_FILE}"

OUTPUT_PREFIX=$(basename "${VCF_FILE}" .vcf.gz)
echo "OUTPUT_PREFIX: ${OUTPUT_PREFIX}"

dx run swiss-army-knife \
        -iin="${VCF_PATH}" \
        -iin="/Bulk/Exome sequences/Exome OQFE CRAM files/helper_files/GRCh38_full_analysis_set_plus_decoy_hla.fa" \
        -iin="/Bulk/Exome sequences/Exome OQFE CRAM files/helper_files/GRCh38_full_analysis_set_plus_decoy_hla.fa.fai" \
        -icmd="bcftools norm ${VCF_FILE} -f GRCh38_full_analysis_set_plus_decoy_hla.fa -m -any -Oz --no-version --threads 4 -o ${OUTPUT_PREFIX}.norm_split.vcf.gz" \
        --name="bcftools_norm_${VCF_FILE}" \
        --instance-type="mem1_ssd1_v2_x4" \
        --destination="${OUTPUT_FOLDER}" \
        --yes \
        --brief \
        --priority high

**Runtime:** ~5m <br>
**Cost:** ~£0.01

#### 3. Reformat variant ID column
After normalisation and splitting, you may end up with multiple variants at the same position that share the same variant ID. In addition, it is likely that some of these variant IDs will be quite long and exceed plink/plink2's 16,000-character limit for variant IDs.

To remedy any issues caused by duplicate variant IDs or exceedingly long variant IDs we can rename, shorten or strip the variant IDs using the code below.

##### 3.1. Shorten variant IDs
When working with VCF files, variant IDs are often of the form "CHROM_POS_REF_ALT". Before normalisation and splitting, multiallelic variants can generate extremely long IDs due to the many possible alternate alleles. After normalisation and splitting, these variants are converted into biallelic records, but the IDs remain unnecessarily long as they retain the IDs from the multiallelic record. To regenerate these variant IDs in the standard "CHROM_POS_REF_ALT" format for the biallelic variants, you can use the following code:

In [ ]:
# shorten IDs to CHROM_POS_ALT_REF 

# output from previous jobs stored in output_folder/
VCF_PATH=$(dx find data --name="*.norm_split.vcf.gz" --path="output_folder" --delimiter=";" | cut -f4 -d ';') 
echo "VCF_PATH: ${VCF_PATH}"

VCF_FILE=$(basename ${VCF_PATH})
echo "VCF_FILE: ${VCF_FILE}"

OUTPUT_PREFIX=$(basename "${VCF_FILE}" .vcf.gz)
echo "OUTPUT_PREFIX: ${OUTPUT_PREFIX}"

dx run swiss-army-knife \
        -iin="${VCF_PATH}" \
        -icmd="bcftools annotate --set-id '%CHROM\_%POS\_%REF\_%ALT' -Oz -o ${OUTPUT_PREFIX}.short_IDs.vcf.gz ${VCF_FILE}" \
        --name="shorten_variant_IDs_${VCF_FILE}" \
        --instance-type="mem1_ssd1_v2_x4" \
        --destination="${OUTPUT_FOLDER}" \
        --yes \
        --brief \
        --priority high

**Runtime:** ~5m <br>
**Cost:** ~£0.01

##### 3.2 Strip variant IDs (replace variant IDs with ".")

Alternatively we can strip the variant IDs using:

In [ ]:
# strip IDs, i.e. replace with "." 

VCF_PATH=$(dx find data --name="*.norm_split.vcf.gz" --path="output_folder" --delimiter=";" | cut -f4 -d ';') 
echo "VCF_PATH: ${VCF_PATH}"

VCF_FILE=$(basename ${VCF_PATH})
echo "VCF_FILE: ${VCF_FILE}"

OUTPUT_PREFIX=$(basename "${VCF_FILE}" .vcf.gz)
echo "OUTPUT_PREFIX: ${OUTPUT_PREFIX}"

dx run swiss-army-knife \
        -iin="${VCF_PATH}" \
        -icmd="bcftools annotate --set-id '.' -Oz -o ${OUTPUT_PREFIX}.no_IDs.vcf.gz ${VCF_FILE}" \
        --name="strip_variant_IDs_${VCF_FILE}" \
        --instance-type="mem1_ssd1_v2_x4" \
        --destination="${OUTPUT_FOLDER}" \
        --yes \
        --brief \
        --priority high

**Runtime:** ~5m <br>
**Cost:** ~£0.01

#### 4. Conversion of phased pVCF to plink2 binary format 
There may be cases where you want to convert a pVCF to plink2 format (.pgen/.pvar/.psam) to ensure compatibility with specific tools, such as those used in GWAS, population genetics, or polygenic risk scoring analyses. To do this we use the code below with the following arguments:
- --keep-allele-order: preserves the reference/alternate allele order from the VCF file.
- --vcf-idspace-to _ : converts spaces in variant IDs to underscores (\_).
- --double-id: duplicates the sample ID in both the Family ID (FID) and Individual ID (IID) fields.
- --allow-extra-chr 0: allows plink2 to process chromosomes that are not standard autosomes (1–22).
- --vcf-half-call m: tells plink2 to treat half calls as missing.

In [ ]:
# convert pVCF to plink2 format

# here we will use the pVCF file with shortened variant IDs
VCF_PATH=$(dx find data --name="*.short_IDs.vcf.gz" --path="output_folder" --delimiter=";" | cut -f4 -d ';') 
echo "VCF_PATH: ${VCF_PATH}"

VCF_FILE=$(basename ${VCF_PATH})
echo "VCF_FILE: ${VCF_FILE}"

OUTPUT_PREFIX=$(basename "${VCF_FILE}" .vcf.gz)
echo "OUTPUT_PREFIX: ${OUTPUT_PREFIX}"

dx run swiss-army-knife \
        -iin="${VCF_PATH}" \
        -icmd="plink2 --make-pgen --vcf ${VCF_FILE} --keep-allele-order --vcf-idspace-to _ --double-id --allow-extra-chr 0 --vcf-half-call m --out ${OUTPUT_PREFIX}" \
        --name="convert_vcf_plink2_${VCF_FILE}" \
        --instance-type="mem1_ssd1_v2_x4" \
        --destination="${OUTPUT_FOLDER}" \
        --yes \
        --brief \
        --priority high

**Runtime:** ~5m <br>
**Cost:** ~£0.01

#### 5. Conversion of phased pVCF to BGEN format 
You might also want to convert the pVCF file to BGEN format (.bgen/.sample) to improve data storage efficiency or to use software tools that require BGEN input, such as BGENIE or Hail. To do this using plink2, you must first convert the VCF file to plink2 format (see above) and then from plink2 format to BGEN using:

In [ ]:
# convert plink2 format to BGEN format

VCF_PATH=$(dx find data --name="*.filtered_samples_regions.norm_split.short_IDs.vcf.gz" --path="output_folder" --delimiter=";" | cut -f4 -d ';') 
echo "VCF_PATH: ${VCF_PATH}"

VCF_FILE=$(basename ${VCF_PATH})
echo "VCF_FILE: ${VCF_FILE}"

# remove .vcf.gz (string substitution)
VCF_PREFIX="${VCF_PATH%.vcf.gz}"
echo "VCF_PREFIX: ${VCF_PREFIX}"

OUTPUT_PREFIX=$(basename "${VCF_FILE}" .vcf.gz)
echo "OUTPUT_PREFIX: ${OUTPUT_PREFIX}"

dx run swiss-army-knife \
        -iin="${VCF_PREFIX}.pgen" \
        -iin="${VCF_PREFIX}.pvar" \
        -iin="${VCF_PREFIX}.psam" \
        -icmd="plink2 --pfile ${OUTPUT_PREFIX} --export bgen-1.2 bits=8 ref-first --out ${OUTPUT_PREFIX}" \
        --name="convert_plink2_bgen_${OUTPUT_PREFIX}" \
        --instance-type="mem1_ssd1_v2_x4" \
        --destination="${OUTPUT_FOLDER}" \
        --yes \
        --brief \
        --priority high

**Runtime:** ~5m
<br>
**Cost:** ~£0.01

#### 6. Submit jobs for multiple VCFs 
Up to this point we have been focusing on running operations on chromosome 11 only, of course you might want to run these operations on more than one pVCF. In these cases, we can create a file containing a list of pVCF files we wish to process and then set a Swiss Army Knife job running for each of these input files.

For example, say we want to convert several pVCF files to plink2 format, these jobs can be launched using the code below. Here we want to submit jobs for chromosome 11, 18 and 21, and we are going to assume that we have previously generated filtered VCF files, with shortened variant IDs, for these chromosomes and stored them in "output_folder/".

In [ ]:
# how to do it over multiple files (e.g. chr 11, 18 and 21) 
# generate list of files to convert (or already have a pre-specified list)
dx find data --name="*.short_IDs.vcf.gz" --path="output_folder" --delimiter=";" | cut -f4 -d ';' | grep -E 'c(11|18|21)' > list_to_convert

In [ ]:
# loop over files to set multiple jobs running at once

while read FILE; 
do 
VCF="${FILE##*/}" 
echo "VCF_FILE: $VCF"

dx run swiss-army-knife \
        -iin="${FILE}" \
        -icmd="plink2 --make-pgen --vcf ${VCF} --keep-allele-order --vcf-idspace-to _ --double-id --allow-extra-chr 0 --vcf-half-call m --out ${VCF%.vcf.gz}" \
        --name="convert_vcf_plink2_${VCF}" \
        --instance-type="mem1_ssd1_v2_x8" \
        --destination="$OUTPUT_FOLDER" \
        --yes \
        --brief \
        --priority high;
        
done < list_to_convert

**Runtime:** ~5m each
<br>
**Cost:** ~£0.01-£0.02 each

#### 7. Merging plink2 files for each pVCF
Once you have converted the pVCF files of interest, e.g. chr 11, 18 and 21, to plink2 format you may want to merge these together into one .pgen file.

In [ ]:
# generate list of pgens to merge (without suffix)
dx ls output_folder/*.pgen | sed 's/\.pgen$//;s/^/\/mnt\/project\/output_folder\//g' > merge_list.txt

In [ ]:
# upload merge_list.txt to project so that it can be used by Swiss Army Knife
dx upload merge_list.txt --path helper_files/

In [ ]:
OUTPUT_NAME="merged_chr11_18_21"
echo "OUTPUT_NAME: ${OUTPUT_NAME}"

dx run swiss-army-knife \
        -iin="/helper_files/merge_list.txt" \
        -icmd="plink2 --pmerge-list merge_list.txt --make-pgen --out ${OUTPUT_NAME} --multiallelics-already-joined --max-alleles 2" \
        --name="merge_pgens" \
        --instance-type="mem2_ssd2_v2_x8" \
        --destination="$OUTPUT_FOLDER" \
        --yes \
        --brief \
        --priority high

**Runtime:** ~5m <br>
**Cost:** ~£0.02